# Egyptian Civil Law — RAG + Gradio UI

Same pipeline as `neo4j_rag_qwen3.ipynb` — wrapped in a **Gradio web app** that opens directly in Colab.

**Tabs inside the app:**
- **Ask** — full RAG: question → answer + source articles (English & Arabic)
- **Search** — retrieval only, no LLM, returns ranked articles

Run all cells top-to-bottom. The last cell prints a public `gradio.live` link.

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install neo4j FlagEmbedding transformers torch tqdm ollama gradio

## Step 2 — Install & Start Ollama, Pull Qwen3:4b

In [ ]:
import subprocess, time, requests

print('Installing Ollama...')
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
               shell=True, capture_output=True)

print('Starting Ollama server...')
ollama_proc = subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(30):
    try:
        if requests.get('http://localhost:11434').status_code == 200:
            print('Ollama server is ready!')
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError('Ollama server did not start. Restart runtime and retry.')

In [ ]:
LLM_MODEL = 'qwen3:4b'

print(f'Pulling {LLM_MODEL}  (2.5 GB — takes ~3 min on Colab)...')
!ollama pull qwen3:4b
print('Model ready.')

## Step 3 — Neo4j Connection

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = 'neo4j+s://785ea338.databases.neo4j.io'
NEO4J_USER     = '785ea338'
NEO4J_PASSWORD = '2e3Vah_a8qA5Q14DaECSa87pj0LifbWK_sJq6kZWbsE'
NEO4J_DATABASE = '785ea338'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j Aura!')

## Step 4 — Load BGE-M3

In [ ]:
from FlagEmbedding import BGEM3FlagModel

print('Loading BGE-M3  (first run downloads ~2 GB)...')
embed_model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
print('BGE-M3 ready.')

def embed(text: str) -> list:
    out = embed_model.encode([text], batch_size=1, max_length=512)
    return out['dense_vecs'][0].tolist()

## Step 5 — RAG Pipeline Functions

Identical logic to `neo4j_rag_qwen3.ipynb` — plus a `rag_full()` that returns a
structured dict so the Gradio UI can display answer, sources, and metadata separately.

In [ ]:
import ollama, json, re

# ── Prompts ────────────────────────────────────────────────────────────────

EXTRACT_SYSTEM = """\
You are a metadata extractor for an Egyptian Civil Law database.
Given the user's question (English or Arabic), extract search metadata.

Return ONLY a valid JSON object — no markdown fences, no extra text:
{
  "keywords_en": ["list", "of", "english", "legal", "keywords"],
  "keywords_ar": ["قائمة", "الكلمات", "العربية"],
  "legal_topics": ["broad legal topics, e.g. prescription, contracts, property"],
  "article_numbers": [],
  "search_query": "one refined sentence capturing the core legal question"
}

Rules:
- article_numbers: list of integers if the question mentions specific articles, else []
- keywords_en / keywords_ar: 3-8 terms each, only what is clearly in the question
- legal_topics: 1-4 broad categories
- search_query: English even if the question is in Arabic
"""

ANSWER_SYSTEM = """\
You are a highly knowledgeable legal assistant specializing in Egyptian Civil Law.

You will be given:
1. A question from the user (English or Arabic)
2. A set of relevant law articles retrieved from the database, each with Arabic and English text

Your task:
- Answer the question accurately based ONLY on the provided articles
- Cite the article number(s) you relied on (e.g., "According to Article 7...")
- If the articles do not contain enough information, say so clearly
- If the question is in Arabic, answer in Arabic; if in English, answer in English
- Be precise and professional — this is a legal context
"""

# ── Neo4j helper ───────────────────────────────────────────────────────────

def _run(q, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return s.run(q, **params).data()

# ── Stage 1: Metadata extraction ───────────────────────────────────────────

def extract_metadata(question: str) -> dict:
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': EXTRACT_SYSTEM},
            {'role': 'user',   'content': '/no_think\n' + question},
        ],
        format='json',
        options={'temperature': 0.0},
    )
    raw = re.sub(r'<think>.*?</think>', '', response['message']['content'],
                 flags=re.DOTALL).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'keywords_en': question.split()[:5], 'keywords_ar': [],
                'legal_topics': [], 'article_numbers': [],
                'search_query': question}

# ── Stage 2: Retrieval ─────────────────────────────────────────────────────

def fetch_by_number(numbers: list) -> list:
    if not numbers:
        return []
    rows = _run(
        'MATCH (a:Article) WHERE a.number IN $nums '
        'RETURN a.id AS id, a.number AS number, '
        '       a.english AS english, a.arabic AS arabic',
        nums=numbers
    )
    return [{'id': r['id'], 'number': r['number'],
             'english': r['english'], 'arabic': r['arabic'],
             'kw_score': 1.0, 'sem_score': 1.0, 'source': 'direct'}
            for r in rows]

def fetch_by_keywords(keywords_en, keywords_ar, legal_topics, limit=15) -> list:
    all_kw = keywords_en + keywords_ar + legal_topics
    if not all_kw:
        return []
    kw_rows = _run("""
        UNWIND $keywords AS kw
        MATCH (a:Article)-[:HAS_KEYWORD]->(k:Keyword)
        WHERE toLower(k.name) CONTAINS toLower(kw)
        WITH  a, count(DISTINCT k) AS hits
        ORDER BY hits DESC LIMIT $limit
        RETURN a.id AS id, a.number AS number,
               a.english AS english, a.arabic AS arabic, hits AS kw_hits
    """, keywords=all_kw, limit=limit)
    sec_rows = _run("""
        UNWIND $topics AS topic
        MATCH (s:Section)<-[:IN_SECTION]-(a:Article)
        WHERE toLower(s.name) CONTAINS toLower(topic)
        WITH  a, count(DISTINCT s) AS hits
        ORDER BY hits DESC LIMIT $limit
        RETURN a.id AS id, a.number AS number,
               a.english AS english, a.arabic AS arabic, hits AS kw_hits
    """, topics=legal_topics or keywords_en or [''], limit=limit)
    merged = {}
    for r in kw_rows + sec_rows:
        aid = r['id']
        if aid not in merged or r['kw_hits'] > merged[aid]['kw_hits']:
            merged[aid] = r
    max_hits = max((v['kw_hits'] for v in merged.values()), default=1)
    return [{'id': v['id'], 'number': v['number'],
             'english': v['english'], 'arabic': v['arabic'],
             'kw_score': v['kw_hits'] / max_hits, 'sem_score': 0.0,
             'source': 'keyword'} for v in merged.values()]

def fetch_by_semantic(query_text: str, top_k=15) -> list:
    vec  = embed(query_text)
    rows = _run("""
        CALL db.index.vector.queryNodes('article_embedding', $topK, $vec)
        YIELD node AS a, score
        RETURN a.id AS id, a.number AS number,
               a.english AS english, a.arabic AS arabic, score AS sem_score
    """, vec=vec, topK=top_k)
    return [{'id': r['id'], 'number': r['number'],
             'english': r['english'], 'arabic': r['arabic'],
             'kw_score': 0.0, 'sem_score': float(r['sem_score']),
             'source': 'semantic'} for r in rows]

# ── Stage 3: Reranking ─────────────────────────────────────────────────────

def rerank(direct, keyword, semantic, top_k=5) -> list:
    pool = {}
    def add(records, field, weight):
        for r in records:
            aid = r['id']
            if aid not in pool:
                pool[aid] = {'id': r['id'], 'number': r['number'],
                             'english': r['english'], 'arabic': r['arabic'],
                             'score': 0.0, 'source': r.get('source', '')}
            pool[aid]['score'] += r.get(field, 0.0) * weight
    for r in direct:
        r['kw_score'] = r['sem_score'] = 1.0
    add(direct,   'sem_score', 0.55); add(direct,   'kw_score', 0.35)
    add(keyword,  'kw_score',  0.35); add(keyword,  'sem_score', 0.55)
    add(semantic, 'sem_score', 0.55); add(semantic, 'kw_score', 0.35)
    for r in direct:
        if r['id'] in pool: pool[r['id']]['score'] += 0.10
    return sorted(pool.values(), key=lambda x: x['score'], reverse=True)[:top_k]

# ── Stage 4: Answer generation ─────────────────────────────────────────────

def generate_answer(question: str, articles: list) -> str:
    if not articles:
        return 'No relevant articles were found in the database for this question.'
    context = '\n\n'.join(
        f"--- {a['id']} (score={a['score']:.3f}) ---\n"
        f"[English]\n{a['english']}\n\n[Arabic]\n{a['arabic']}"
        for a in articles
    )
    response = ollama.chat(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': ANSWER_SYSTEM},
            {'role': 'user',   'content': f'RETRIEVED LAW ARTICLES:\n\n{context}\n\n---\n\nQUESTION: {question}'},
        ],
        options={'temperature': 0.3},
    )
    raw = response['message']['content']
    return re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()

# ── Full pipeline (returns dict for Gradio) ────────────────────────────────

def rag_full(question: str, top_k: int = 5) -> dict:
    """Returns answer + articles + metadata as a dict."""
    meta     = extract_metadata(question)
    direct   = fetch_by_number(meta.get('article_numbers', []))
    keyword  = fetch_by_keywords(meta.get('keywords_en', []),
                                  meta.get('keywords_ar', []),
                                  meta.get('legal_topics', []))
    semantic = fetch_by_semantic(meta.get('search_query', question))
    top      = rerank(direct, keyword, semantic, top_k=top_k)
    answer   = generate_answer(question, top)
    return {'answer': answer, 'articles': top, 'metadata': meta}

print('RAG pipeline ready.')

## Step 6 — Gradio UI

Two tabs:
- **Ask** — full RAG pipeline with answer + source articles
- **Search** — retrieval only, ranked article cards

In [ ]:
import gradio as gr

# ── HTML renderers ─────────────────────────────────────────────────────────

def articles_to_html(articles: list) -> str:
    """Renders source articles as expandable HTML cards."""
    if not articles:
        return '<p style="color:#888">No articles retrieved.</p>'

    SOURCE_COLORS = {'direct': '#2563eb', 'keyword': '#16a34a', 'semantic': '#9333ea'}

    cards = []
    for art in articles:
        color   = SOURCE_COLORS.get(art.get('source', ''), '#64748b')
        en_text = (art.get('english') or '').replace('\n', '<br>')
        ar_text = (art.get('arabic')  or '').replace('\n', '<br>')
        cards.append(f"""
        <details style="margin:10px 0; border:1px solid #e2e8f0;
                        border-radius:10px; overflow:hidden;">
          <summary style="padding:12px 16px; cursor:pointer;
                          background:#f8fafc; font-weight:600;
                          display:flex; align-items:center; gap:10px;">
            <span style="background:{color}; color:#fff; padding:2px 10px;
                         border-radius:20px; font-size:13px;">{art['id']}</span>
            <span style="color:#475569">Score: {art.get('score', 0):.3f}</span>
            <span style="margin-left:auto; background:#f1f5f9;
                         padding:2px 8px; border-radius:12px; font-size:12px;
                         color:#64748b;">{art.get('source','')}</span>
          </summary>
          <div style="padding:16px;">
            <p style="font-size:13px; color:#1e293b; margin:0 0 12px 0;">
              <strong>English</strong><br>{en_text[:600]}{'…' if len(en_text)>600 else ''}
            </p>
            <p style="font-size:13px; color:#1e293b; margin:0;
                       direction:rtl; text-align:right; font-family:serif;">
              <strong>Arabic</strong><br>{ar_text[:600]}{'…' if len(ar_text)>600 else ''}
            </p>
          </div>
        </details>
        """)
    return ''.join(cards)


def metadata_to_html(meta: dict) -> str:
    """Renders extracted metadata as a compact info block."""
    if not meta:
        return ''
    rows = [
        ('Keywords EN',  ', '.join(meta.get('keywords_en', []))  or '—'),
        ('Keywords AR',  ', '.join(meta.get('keywords_ar', []))  or '—'),
        ('Legal topics', ', '.join(meta.get('legal_topics', [])) or '—'),
        ('Article #s',   ', '.join(map(str, meta.get('article_numbers', []))) or '—'),
        ('Search query', meta.get('search_query', '—')),
    ]
    inner = ''.join(
        f'<tr><td style="padding:4px 12px 4px 0;color:#64748b;white-space:nowrap;"\
            ><strong>{k}</strong></td>'
        f'<td style="padding:4px 0;color:#1e293b">{v}</td></tr>'
        for k, v in rows
    )
    return f'<table style="font-size:13px;border-collapse:collapse">{inner}</table>'


print('Helper functions ready.')

In [ ]:
# ── Gradio event handlers ──────────────────────────────────────────────────

def handle_ask(question: str, top_k: int):
    """
    Returns:
      answer_md      – markdown answer
      sources_html   – HTML article cards
      meta_html      – HTML metadata table
    """
    question = question.strip()
    if not question:
        return '⚠️ Please enter a question.', '', ''
    try:
        result = rag_full(question, top_k=int(top_k))
        return (
            result['answer'],
            articles_to_html(result['articles']),
            metadata_to_html(result['metadata']),
        )
    except Exception as e:
        return f'❌ Error: {e}', '', ''


def handle_search(query: str, top_k: int):
    """
    Returns:
      results_html – HTML article cards
      meta_html    – HTML metadata table
    """
    query = query.strip()
    if not query:
        return '<p style="color:#888">Please enter a search query.</p>', ''
    try:
        meta     = extract_metadata(query)
        direct   = fetch_by_number(meta.get('article_numbers', []))
        keyword  = fetch_by_keywords(meta.get('keywords_en', []),
                                      meta.get('keywords_ar', []),
                                      meta.get('legal_topics', []))
        semantic = fetch_by_semantic(meta.get('search_query', query))
        articles = rerank(direct, keyword, semantic, top_k=int(top_k))
        return articles_to_html(articles), metadata_to_html(meta)
    except Exception as e:
        return f'<p style="color:red">Error: {e}</p>', ''


print('Event handlers ready.')

In [ ]:
# ── Gradio Blocks UI ───────────────────────────────────────────────────────

CSS = """
.answer-box { font-size: 15px; line-height: 1.7; }
.label-bold { font-weight: 700; color: #1e293b; }
footer { display: none !important; }
"""

EXAMPLES_ASK = [
    ["What are the conditions under which exercising a right becomes unlawful?", 5],
    ["ما هي القواعد المتعلقة بالتقادم وانقضاء المدد الزمنية في القانون المدني؟", 5],
    ["Explain Article 5 and how it relates to Article 4", 5],
    ["What happens when a law is repealed by a new legislation?", 5],
]

EXAMPLES_SEARCH = [
    ["prescription and limitation periods", 8],
    ["legal capacity and persons", 8],
    ["contracts obligations", 8],
]

with gr.Blocks(css=CSS, title='Egyptian Civil Law Assistant',
               theme=gr.themes.Soft(primary_hue='blue')) as demo:

    # ── Header ──────────────────────────────────────────────────────────────
    gr.Markdown("""
    # 🏛 Egyptian Civil Law Assistant
    **Ask questions in English or Arabic** — powered by BGE-M3 embeddings,
    Neo4j knowledge graph, and Qwen3:4b (Ollama).
    """)

    # ── Tab 1: Ask ───────────────────────────────────────────────────────────
    with gr.Tab('💬 Ask a Question'):

        with gr.Row():

            # Left column — input
            with gr.Column(scale=1, min_width=300):
                question_box = gr.Textbox(
                    label='Your Question',
                    placeholder='Ask in English or Arabic …',
                    lines=4,
                    max_lines=8,
                )
                top_k_ask = gr.Slider(
                    minimum=1, maximum=10, value=5, step=1,
                    label='Source articles to retrieve',
                )
                ask_btn = gr.Button('Ask', variant='primary', size='lg')

                with gr.Accordion('Extracted Metadata', open=False):
                    meta_ask = gr.HTML(label='')

                gr.Examples(
                    examples=EXAMPLES_ASK,
                    inputs=[question_box, top_k_ask],
                    label='Example questions',
                )

            # Right column — output
            with gr.Column(scale=2):
                answer_box = gr.Markdown(
                    label='Answer',
                    value='*Your answer will appear here …*',
                    elem_classes=['answer-box'],
                )
                with gr.Accordion('📄 Source Articles', open=True):
                    sources_ask = gr.HTML()

        ask_btn.click(
            fn=handle_ask,
            inputs=[question_box, top_k_ask],
            outputs=[answer_box, sources_ask, meta_ask],
        )
        # Also trigger on Shift+Enter in the textbox
        question_box.submit(
            fn=handle_ask,
            inputs=[question_box, top_k_ask],
            outputs=[answer_box, sources_ask, meta_ask],
        )

    # ── Tab 2: Search ────────────────────────────────────────────────────────
    with gr.Tab('🔍 Search Articles'):

        gr.Markdown('Retrieval only — no answer generated. Useful for browsing related articles.')

        with gr.Row():
            search_box = gr.Textbox(
                label='Search Query',
                placeholder='Enter keywords or a legal concept …',
                lines=2,
            )
            top_k_search = gr.Slider(
                minimum=1, maximum=20, value=8, step=1,
                label='Max results',
            )

        search_btn = gr.Button('Search', variant='primary')

        with gr.Accordion('Extracted Metadata', open=False):
            meta_search = gr.HTML()

        search_results = gr.HTML(label='Results')

        gr.Examples(
            examples=EXAMPLES_SEARCH,
            inputs=[search_box, top_k_search],
            label='Example queries',
        )

        search_btn.click(
            fn=handle_search,
            inputs=[search_box, top_k_search],
            outputs=[search_results, meta_search],
        )
        search_box.submit(
            fn=handle_search,
            inputs=[search_box, top_k_search],
            outputs=[search_results, meta_search],
        )

print('Gradio app defined — run the next cell to launch.')

## Step 7 — Launch the App

`share=True` creates a **public `gradio.live` URL** — Colab does not support
localhost, so this is required. The link is printed below the cell.

In [ ]:
demo.launch(
    share=True,          # generates a public gradio.live link valid for 72 h
    debug=False,         # set True to see stack traces in the cell output
    show_error=True,
    quiet=False,
)